# SOLSTICE quickstart: first DIII-D state models

Train a baseline state model (control parameters -> 2D plasma fields) on the
DIII-D APP-FPP ensemble store (`solstice_store_diiid_appfpp_v0.nc`, 761 SOLPS-ITER
cases on the native 96x36 grid, built with `solstice.data.store`).

Upload the store file (or mount Drive/Dropbox) and set `STORE` below.
Runs on a free Colab GPU or CPU.


In [ ]:
!pip -q install xarray netcdf4
STORE = 'solstice_store_diiid_appfpp_v0.nc'  # path to the store file


In [ ]:
import numpy as np, xarray as xr, torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

ds = xr.open_dataset(STORE)
print(dict(ds.sizes))
INPUTS = sorted(v for v in ds.data_vars if v.startswith('input_'))
print('inputs:', INPUTS)


## Look at one case


In [ ]:
def plot_field(values, title='', ax=None, cmap='viridis', log=False):
    verts = np.stack([ds.cell_corners_r.values, ds.cell_corners_z.values], axis=-1)
    v = np.log10(np.clip(values, 1e-30, None)) if log else values
    ax = ax or plt.subplots(figsize=(4, 6))[1]
    pc = PolyCollection(verts, array=v, cmap=cmap, edgecolor='none')
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal')
    ax.set_xlabel('R (m)'); ax.set_ylabel('Z (m)'); ax.set_title(title)
    plt.colorbar(pc, ax=ax, shrink=0.8)
    return ax

k = 0
fig, axs = plt.subplots(1, 2, figsize=(9, 6))
plot_field(ds.te.isel(case=k).values, 'Te (eV)', axs[0])
plot_field(ds.ne.isel(case=k).values, 'log10 ne (m^-3)', axs[1], log=True)
plt.tight_layout()


## Tensors and normalization

Inputs are log-scaled where they span decades, then standardized.
Fields use log10 for densities and **per-cell** standardization
(location-dependent normalization -- targets span orders of magnitude).


In [ ]:
FIELDS = {'te': True, 'ti': True, 'ne': True, 'ua_D1': False}  # name -> log10?
LOG_INPUTS = ('input_n_core', 'input_puff_D2', 'input_puff_Ne', 'input_dna')

X = np.stack([ds[v].values for v in INPUTS], axis=1).astype(np.float64)
for j, v in enumerate(INPUTS):
    if v in LOG_INPUTS:
        X[:, j] = np.log10(np.clip(X[:, j], 1e-30, None))
x_mean, x_std = X.mean(0), X.std(0) + 1e-12
Xn = (X - x_mean) / x_std

def prep_field(name, log):
    y = ds[name].values.astype(np.float64)
    if log:
        y = np.log10(np.clip(np.abs(y), 1e-6, None))
    mean, std = y.mean(0), y.std(0) + 1e-12   # per-cell stats
    return (y - mean) / std, mean, std

# QC columns (qc_pass, puff_record_missing, params_converged) are advisory:
# all 761 cases currently pass; filter here if that ever changes, e.g.:
# keep = (ds.qc_pass & ~ds.puff_record_missing).values

rng = np.random.default_rng(0)
idx = rng.permutation(ds.sizes['case'])
split = int(0.85 * len(idx))
itr, ite = idx[:split], idx[split:]
print(len(itr), 'train /', len(ite), 'test cases')


## Baseline MLP per field (params -> full 2D field)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_field(name, log, hidden=512, epochs=800, lr=1e-3):
    Yn, mean, std = prep_field(name, log)
    xt = torch.tensor(Xn[itr], dtype=torch.float32, device=device)
    yt = torch.tensor(Yn[itr], dtype=torch.float32, device=device)
    xv = torch.tensor(Xn[ite], dtype=torch.float32, device=device)
    yv = torch.tensor(Yn[ite], dtype=torch.float32, device=device)
    model = nn.Sequential(
        nn.Linear(len(INPUTS), hidden), nn.GELU(),
        nn.Linear(hidden, hidden), nn.GELU(),
        nn.Linear(hidden, ds.sizes['cell'])).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state = np.inf, None
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = nn.functional.mse_loss(model(xt), yt)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val = nn.functional.mse_loss(model(xv), yv).item()
        if val < best:
            best, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
        if ep % 100 == 0:
            print(f'{name} ep{ep}: train {loss.item():.4f} val {val:.4f}')
    model.load_state_dict(best_state)
    return model, mean, std, best

models = {}
for name, log in FIELDS.items():
    models[name] = train_field(name, log)
    print(name, 'best val MSE (normalized):', models[name][3])


## Evaluate: 2D maps and outer-midplane profile


In [ ]:
def predict(name, case_idx):
    model, mean, std, _ = models[name]
    with torch.no_grad():
        yn = model(torch.tensor(Xn[[case_idx]], dtype=torch.float32, device=device)).cpu().numpy()[0]
    y = yn * std + mean
    return 10**y if FIELDS[name] else y

k = int(ite[0])
truth = ds.te.isel(case=k).values
pred = predict('te', k)
fig, axs = plt.subplots(1, 3, figsize=(13, 6))
plot_field(truth, 'Te SOLPS (eV)', axs[0])
plot_field(pred, 'Te SOLSTICE-MLP (eV)', axs[1])
plot_field(100*(pred-truth)/np.clip(truth,1e-3,None), 'error (%)', axs[2], cmap='RdBu_r')
plt.tight_layout()


In [ ]:
# outer-midplane radial profile: pick the poloidal column nearest Z=0 on the LFS
ix_omp = int(ds.cell_ix.values[np.argmin(np.abs(ds.cell_z.values) + (ds.cell_r.values < ds.cell_r.values.mean())*10)])
sel = ds.cell_ix.values == ix_omp
order = np.argsort(ds.cell_iy.values[sel])
r = ds.cell_r.values[sel][order]
plt.figure(figsize=(6, 4))
plt.plot(r, truth[sel][order], 'o-', label='SOLPS')
plt.plot(r, pred[sel][order], 's--', label='MLP')
plt.xlabel('R (m)'); plt.ylabel('Te (eV)'); plt.title(f'OMP profile, case {ds.case.values[k]}')
plt.legend(); plt.yscale('log')


## Save the models

These weights + normalization stats are the ingredients of a SOLSTICE
checkpoint bundle (`docs/specs/checkpoint_spec.md`); packaging comes next.


In [ ]:
import os
os.makedirs('weights', exist_ok=True)
for name, (model, mean, std, _) in models.items():
    torch.save({'state_dict': model.state_dict(),
                'cell_mean': mean, 'cell_std': std, 'log10': FIELDS[name],
                'inputs': INPUTS, 'x_mean': x_mean, 'x_std': x_std},
               f'weights/diiid-appfpp-state-mlp-{name}.pt')
print(os.listdir('weights'))


## Copy the weights to Drive

Colab VMs are ephemeral — copy the trained weights to Drive so they
can be packaged into SOLSTICE checkpoint bundles afterwards.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/SOLPS_DATA/solstice_weights/v1'
!mkdir -p {DEST} && cp -r weights/* {DEST}/
!ls -la {DEST}
